<a href="https://colab.research.google.com/github/martinhdezpacheco/tfg-scraping-madrid/blob/main/extraer_inmueble_tecnocasa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4

In [2]:
import requests
from bs4 import BeautifulSoup
import json
import html
import time
import csv
import os

In [3]:
"""EJEMPLO DE EXTRACCIÓN DE DATOS DE UNA FICHA DE INMUEBLE EN TECNOCASA"""

def extraer_datos_inmueble(url):
    """
    Descarga y extrae los datos de una única ficha de inmueble de Tecnocasa.

    Parámetro:
        url (str): URL completa de la ficha, ej:
                   "https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html"

    Devuelve:
        dict con los campos limpios, o None si algo falló.
    """

    # --- PASO 1: Descargar el HTML crudo ---
    # Añadimos un "User-Agent" que simula un navegador real. En
    # books.toscrape.com no hacía falta porque es una web de práctica
    # sin ninguna protección, pero en una web real como Tecnocasa,
    # si no lo mandamos, el servidor puede rechazar la petición o
    # devolver una versión distinta de la página (o directamente un error).
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        respuesta = requests.get(url, headers=headers, timeout=10)
        respuesta.raise_for_status()  # lanza un error si la petición falló (404, 500...)
    except requests.exceptions.RequestException as e:
        print(f"Error al descargar {url}: {e}")
        return None

    # --- PASO 2: Parsear el HTML con BeautifulSoup ---
    soup = BeautifulSoup(respuesta.text, "html.parser")

    # --- PASO 3: Encontrar la etiqueta <estate-show-v2> ---
    # Aunque no es una etiqueta HTML "de verdad" (es un componente Vue.js
    # personalizado), BeautifulSoup la trata exactamente igual que
    # cualquier otra etiqueta (<div>, <span>, etc.) porque para
    # BeautifulSoup, cualquier palabra entre < > es una etiqueta válida.
    tag_estate = soup.find("estate-show-v2")

    if tag_estate is None:
        print(f"No se encontró <estate-show-v2> en {url}")
        return None

    # --- PASO 4: Extraer el atributo ":estate" ---
    # Este atributo empieza por ":" (es la sintaxis de Vue.js para un
    # "binding dinámico"). Se accede igual que a cualquier otro atributo,
    # con .get("nombre_del_atributo").
    json_crudo = tag_estate.get(":estate")

    if json_crudo is None:
        print(f"No se encontró el atributo :estate en {url}")
        return None

    # --- PASO 5: "Desescapar" el texto ---
    # El HTML convierte las comillas dobles en &quot; para poder meter
    # un JSON entero dentro de un atributo HTML sin romper la sintaxis.
    # html.unescape() deshace esa conversión (&quot; -> ", &amp; -> &, etc.)
    json_texto = html.unescape(json_crudo)

    # --- PASO 6: Convertir el texto en un diccionario de Python ---
    try:
        datos_completos = json.loads(json_texto)
    except json.JSONDecodeError as e:
        print(f"Error al parsear JSON en {url}: {e}")
        return None

    # --- PASO 7: Quedarnos solo con los campos que nos interesan ---
    # datos_completos es un diccionario ENORME (incluye fotos, colegios
    # cercanos, farmacias, datos de la agencia, hipoteca...). Nosotros
    # solo necesitamos los campos relevantes para el dataset de precios.
    #
    # Usamos .get() en vez de [] porque .get() no da error si la clave
    # no existe (devuelve None) - útil porque no todas las fichas tienen
    # rellenos todos los campos.
    # Nota sobre "features.elevator" y campos similares (garden, concierge...):
    # cuando el JSON trae "" (cadena vacía) NO sabemos con certeza si significa
    # "no tiene" o "no se especificó" en el anuncio. Por seguridad, tratamos
    # cualquier valor vacío como dato AUSENTE (None), no como "no" - así evitamos
    # meter un sesgo falso en el modelo. Más adelante, con más fichas descargadas,
    # se puede revisar si esto se puede refinar comparando con el texto libre
    # de la descripción.
    inmueble = {
        "id": datos_completos.get("id"),
        "detail_url": datos_completos.get("detail_url"),
        "tipo": datos_completos.get("type", {}).get("title"),
        "distrito": datos_completos.get("district", {}).get("title"),
        "barrio": datos_completos.get("quarter"),
        "precio": datos_completos.get("numeric_price"),                       # ej: 1399000
        "m2": datos_completos.get("numeric_surface"),                         # ej: "160.00"
        "habitaciones": datos_completos.get("rooms"),                         # ej: "3 dorm." (a limpiar con regex)
        "banos": datos_completos.get("bathrooms"),                            # ej: "3 baños" (a limpiar con regex)
        "planta": datos_completos.get("features", {}).get("floor"),
        "anio_construccion": datos_completos.get("features", {}).get("build_year"),
        "anio_reforma": datos_completos.get("features", {}).get("renovation_year"),
        "categoria": datos_completos.get("features", {}).get("category"),
        "ascensor": datos_completos.get("features", {}).get("elevator"),
        "balcones": datos_completos.get("features", {}).get("balconies"),
        "terrazas": datos_completos.get("features", {}).get("terraces"),
        "jardin": datos_completos.get("features", {}).get("garden"),
        "calefaccion": datos_completos.get("features", {}).get("heating"),
        "clase_energetica": datos_completos.get("energy_data", {}).get("class"),
        # points_of_interest es una estructura anidada (colegios, farmacias,
        # bares... cada uno con nombre y distancia). No cabe en una sola
        # celda de forma "limpia", así que la guardamos como texto JSON
        # completo por ahora, para procesarla aparte más adelante.
        "points_of_interest": json.dumps(datos_completos.get("points_of_interest"), ensure_ascii=False),
        "title": datos_completos.get("title"),
        "description": datos_completos.get("description"),
    }

    # Normalizamos: cualquier campo que sea cadena vacía ("") lo convertimos
    # en None, para que al guardar en CSV quede en blanco de verdad, en vez
    # de aparecer como texto vacío o como la palabra "None".
    for clave, valor in inmueble.items():
        if valor == "":
            inmueble[clave] = None

    return inmueble


# --- PRUEBA: extraemos un único piso para comprobar que funciona ---
if __name__ == "__main__":
    url_prueba = "https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html"
    resultado = extraer_datos_inmueble(url_prueba)

    if resultado:
        print("Datos extraídos correctamente:\n")
        for campo, valor in resultado.items():
            print(f"{campo}: {valor}")
    else:
        print("No se pudieron extraer los datos.")

Datos extraídos correctamente:

id: 664738
detail_url: https://www.tecnocasa.es/venta/piso/madrid/madrid/664738.html
tipo: Piso
distrito: Las Letras Y Cortes
barrio: Huertas - Cortes
precio: 1399000
m2: 160.00
habitaciones: 3 dorm.
banos: 3 baños
planta: 3 (planta ático)
anio_construccion: 1880
anio_reforma: 2002
categoria: Media
ascensor: None
balcones: None
terrazas: None
jardin: None
calefaccion: centralizada (Radiadores)
clase_energetica: e
points_of_interest: {"public_transport": [{"name": "Antón Martín", "class": "railway", "subclass": "subway", "icon": "subway", "distance": "200 m"}, {"name": "Lavapiés", "class": "railway", "subclass": "subway", "icon": "subway", "distance": "550 m"}, {"name": "Sol", "class": "railway", "subclass": "station", "icon": "station", "distance": "770 m"}, {"name": "Madrid-Puerta de Atocha", "class": "railway", "subclass": "station", "icon": "station", "distance": "930 m"}, {"name": "Atocha - Costanilla Desamparados", "class": "bus", "subclass": "bus_s

In [4]:
"""===================================================================
BLOQUE 0: SUBIR EL CSV DEL DÍA ANTERIOR (si existe)
==================================================================="""

from google.colab import files

print("Si ya tienes un CSV de un día anterior, selecciónalo ahora. Si es la primera vez, pulsa 'Cancelar' o cierra el selector.")
subido = files.upload()

print("Bloque 0 completado.")

Si ya tienes un CSV de un día anterior, selecciónalo ahora. Si es la primera vez, pulsa 'Cancelar' o cierra el selector.


Saving URLs_recolectadas.csv to URLs_recolectadas.csv
Bloque 0 completado.


In [5]:
"""===================================================================
BLOQUE 1: EXTRACCIÓN DE TODAS LAS URLs (estado actual de Tecnocasa)
==================================================================="""

todas_las_urls = []
pagina = 1
ids_pagina_1 = set()

while True:
    if pagina == 1:
        url_listado = "https://www.tecnocasa.es/venta/inmuebles/comunidad-de-madrid/madrid/madrid.html"
    else:
        url_listado = f"https://www.tecnocasa.es/venta/inmuebles/comunidad-de-madrid/madrid/madrid.html/pag-{pagina}"

    respuesta = requests.get(url_listado, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)

    if respuesta.status_code != 200:
        print(f"Página {pagina} no disponible (status {respuesta.status_code}). Fin.")
        break

    soup = BeautifulSoup(respuesta.text, "html.parser")
    tag_estates = soup.find("estates-index")

    if tag_estates is None:
        print(f"No se encontró <estates-index> en la página {pagina}. Fin.")
        break

    json_texto = html.unescape(tag_estates.get(":estates"))
    lista_anuncios = json.loads(json_texto)

    if len(lista_anuncios) == 0:
        print(f"Página {pagina} sin anuncios. Fin.")
        break

    ids_actuales = {anuncio.get("id") for anuncio in lista_anuncios}

    if pagina == 1:
        ids_pagina_1 = ids_actuales
    else:
        coincidencias = len(ids_actuales & ids_pagina_1)
        if coincidencias >= len(ids_actuales) / 2:
            print(f"Página {pagina} parece ser fallback. Fin real del listado.")
            break

    for anuncio in lista_anuncios:
        url_detalle = anuncio.get("detail_url")
        if url_detalle:
            todas_las_urls.append(url_detalle)

    print(f"Página {pagina}: {len(lista_anuncios)} anuncios encontrados (acumulado: {len(todas_las_urls)})")

    pagina += 1
    time.sleep(1)

print(f"Bloque 1 completado. Total de URLs en Tecnocasa hoy: {len(todas_las_urls)}")

Página 1: 15 anuncios encontrados (acumulado: 15)
Página 2: 15 anuncios encontrados (acumulado: 30)
Página 3: 15 anuncios encontrados (acumulado: 45)
Página 4: 15 anuncios encontrados (acumulado: 60)
Página 5: 15 anuncios encontrados (acumulado: 75)
Página 6: 15 anuncios encontrados (acumulado: 90)
Página 7: 15 anuncios encontrados (acumulado: 105)
Página 8: 15 anuncios encontrados (acumulado: 120)
Página 9: 15 anuncios encontrados (acumulado: 135)
Página 10: 15 anuncios encontrados (acumulado: 150)
Página 11: 15 anuncios encontrados (acumulado: 165)
Página 12: 15 anuncios encontrados (acumulado: 180)
Página 13: 15 anuncios encontrados (acumulado: 195)
Página 14: 15 anuncios encontrados (acumulado: 210)
Página 15: 15 anuncios encontrados (acumulado: 225)
Página 16: 15 anuncios encontrados (acumulado: 240)
Página 17: 15 anuncios encontrados (acumulado: 255)
Página 18: 15 anuncios encontrados (acumulado: 270)
Página 19: 15 anuncios encontrados (acumulado: 285)
Página 20: 15 anuncios enco

In [6]:
"""======================================================================
BLOQUE 2: EXTRACCIÓN DE LAS URLs NUEVAS (comparando con el CSV existente)
======================================================================"""

nombre_csv = "URLs_recolectadas.csv"

urls_antiguas = set()
if os.path.exists(nombre_csv):
    with open(nombre_csv, "r", encoding="utf-8-sig") as archivo:
        lector = csv.DictReader(archivo)
        for fila in lector:
            urls_antiguas.add(fila["url_detalle"])

urls_nuevas = set(todas_las_urls) - urls_antiguas

print(f"Bloque 2 completado: {len(urls_antiguas)} URLs antiguas leídas, {len(urls_nuevas)} URLs nuevas detectadas.")

Bloque 2 completado: 961 URLs antiguas leídas, 20 URLs nuevas detectadas.


In [7]:
"""===================================================================
BLOQUE 3: CONTEO — ANTES vs. DESPUÉS DE LA ACTUALIZACIÓN
==================================================================="""

print(f"URLs que había antes de esta ejecución: {len(urls_antiguas)}")
print(f"URLs nuevas encontradas hoy: {len(urls_nuevas)}")
print(f"Total tras la actualización: {len(urls_antiguas) + len(urls_nuevas)}")
print("Bloque 3 completado.")

URLs que había antes de esta ejecución: 961
URLs nuevas encontradas hoy: 20
Total tras la actualización: 981
Bloque 3 completado.


In [8]:
"""===================================================================
BLOQUE 4: EXPORTAR — AÑADIR LAS URLs NUEVAS AL CSV
==================================================================="""

archivo_existe = os.path.exists(nombre_csv)

with open(nombre_csv, "a", newline="", encoding="utf-8-sig") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=["url_detalle"])
    if not archivo_existe:
        escritor.writeheader()
    for url in urls_nuevas:
        escritor.writerow({"url_detalle": url})

print(f"Bloque 4 completado. Se han añadido {len(urls_nuevas)} URLs nuevas a {nombre_csv}")

Bloque 4 completado. Se han añadido 20 URLs nuevas a URLs_recolectadas.csv


In [9]:
"""===================================================================
BLOQUE 5: DESCARGAR EL CSV ACTUALIZADO
==================================================================="""

files.download(nombre_csv)

print("Bloque 5 completado. Guarda este CSV para subirlo mañana en el Bloque 0.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Bloque 5 completado. Guarda este CSV para subirlo mañana en el Bloque 0.
